# LLMs & Transformers
by AI@UCI
# ** ENSURE YOU ARE RUNNING THIS IN AN ENVIRONMENT WITH THE REQUIRED PACKAGES **


In [ ]:
! pip install ? ? ? ?

In [ ]:
import ? as np
import ? as plt
from ? import AutoTokenizer, AutoModel, pipeline

## 1. Tokenization


In [ ]:
# Load the GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained(?)

text = "Large language models are transforming AI research."

# encode text to token IDs
token_ids = tokenizer.?(text)
print("Token IDs:", token_ids)

# convert IDs to readable token strings
tokens = tokenizer.convert_ids_to_tokens(?)
print("Tokens:   ", tokens)

print("\nVocabulary size:", tokenizer.?)

In [ ]:
# decode token IDs back to text
decoded = tokenizer.?(token_ids)
print("Decoded text:", decoded)

# count tokens per word and plot
words = text.split()
word_token_counts = []
for word in words:
    count = len(tokenizer.encode(" " + word, add_special_tokens=False))
    word_token_counts.append(count)

plt.bar(?, ?, color='steelblue')
plt.title("Number of Tokens per Word")
plt.ylabel("Token count")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 2. Embeddings


In [ ]:
import torch

# load the GPT-2 model
model = AutoModel.from_pretrained(?)
embedding_layer = model.?  # word token embeddings attribute

print("Embedding matrix shape:", embedding_layer.weight.shape)
# expected shape: (50257, 768)

# look up embedding for 'hello' (token ID 15496)
sample_embedding = embedding_layer.weight[?].detach().numpy()
print("\nEmbedding vector (first 10 dims):", sample_embedding[:10])
print("Embedding dimension:", len(sample_embedding))

In [ ]:
# cosine similarity: measures how similar two vectors are (range: -1 to 1)
def cosine_similarity(a, b):
    return np.dot(?, ?) / (np.linalg.norm(?) * np.linalg.norm(?))

word_pairs = [("king", "queen"), ("dog", "cat"), ("computer", "table")]

for w1, w2 in word_pairs:
    id1 = tokenizer.encode(" " + w1, add_special_tokens=False)[0]
    id2 = tokenizer.encode(" " + w2, add_special_tokens=False)[0]
    e1 = embedding_layer.weight[id1].detach().numpy()
    e2 = embedding_layer.weight[id2].detach().numpy()
    sim = cosine_similarity(?, ?)
    print(f"Similarity('{w1}', '{w2}'): {sim:.4f}")

## 3. Attention Mechanism

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$


In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    # step 1: compute raw scores (Q dot K-transpose, scaled by sqrt(d_k))
    scores = np.dot(?, ?) / np.sqrt(?)
    # step 2: apply softmax to get attention weights
    weights = ?(scores)
    # step 3: weighted sum of V
    output = np.dot(?, ?)
    return output, weights

In [ ]:
np.random.seed(42)
seq_len, d_k = 4, 3

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, weights = scaled_dot_product_attention(?, ?, ?)

print("Attention weights (each row should sum to 1):")
print(np.round(weights, 3))
print("\nOutput shape:", output.shape)

# heatmap of attention weights
tokens_demo = ["The", "cat", "sat", "down"]
plt.figure(figsize=(5, 4))
plt.imshow(weights, cmap='Blues')
plt.colorbar()
plt.xticks(range(seq_len), tokens_demo)
plt.yticks(range(seq_len), tokens_demo)
plt.title("Attention Weight Heatmap")
plt.xlabel("Keys (attended to)")
plt.ylabel("Queries (attending)")
plt.tight_layout()
plt.show()

## 4. Text Generation with GPT-2


In [ ]:
# create a text-generation pipeline using GPT-2
generator = pipeline(?, model=?)

prompt = "Artificial intelligence is"
output = generator(
    prompt,
    max_new_tokens=?,    # number of new tokens to generate
    do_sample=True,
    temperature=?,       # try 0.2, 0.8, or 1.5
    top_k=?,             # only sample from top-k tokens
    num_return_sequences=?
)

print(output[0]['generated_text'])

In [ ]:
# compare different temperatures
prompt = "The future of machine learning"
temperatures = [?, ?, ?]  # fill in 3 temperatures to compare

for temp in temperatures:
    result = generator(
        prompt,
        max_new_tokens=40,
        do_sample=True,
        temperature=?,
        top_k=50,
        num_return_sequences=1
    )
    print(f"--- Temperature: {temp} ---")
    print(result[0]['generated_text'])
    print()

In [ ]:
# your turn: write your own prompt and generate 2 continuations
my_prompt = ?  # fill in your prompt string

my_output = generator(
    ?,
    max_new_tokens=?,
    do_sample=?,
    temperature=?,
    top_k=?,
    num_return_sequences=?
)

for i, out in enumerate(my_output):
    print(f"--- Continuation {i+1} ---")
    print(out['generated_text'])
    print()